In [2]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle   

data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Preprocess data
data
data= data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [ ]:
#lable Encoding for the Gender(Fit- Find unique values from gender sort them into alphabetical order)
#Transform - Convert the unique text values into numerical valus are zeros and 1 s
label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
##Onehot encoding for the geography
import pickle
onehot_encoder = OneHotEncoder(sparse_output=True)
geo_encoded = onehot_encoder.fit_transform(data[["Geography"]]).toarray()
column_names = onehot_encoder.get_feature_names_out(["Geography"])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=column_names)
data1 = data.drop("Geography", axis=1)
##Combine data
input_data = pd.concat([data1, geo_encoded_df], axis=1)

##Split Data into independent and dependent features
X = input_data.drop("EstimatedSalary", axis=1)
y = input_data["EstimatedSalary"]

##Split data in training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)
##Scale this fearures
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)




In [6]:
##Convert Pickle files
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder, file)
with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(onehot_encoder, file)
with open("scaler.pkl", "wb") as file:
   pickle.dump(scaler, file)
   

##Train the ANN Regression Problem statement 

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [8]:
##Build a Model

model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
        Dense(32, activation="relu"),
        Dense(1),
    ]
)
##Complie the model
model.compile(optimizer="adam", loss="mean_absolute_error", metrics=["mae"])
model.summary()

d:\GenAI2026\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from datetime import datetime

##Setup Tensorboard
log_dir = "regression_logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")

tensor_board_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
##Setup callbacks
early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
##Train the model
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensor_board_callback, early_stopping_callback],
)
#Load Tensorboard
%load_ext tensorboard
%tensorboard --logdir=log_dir
%reload_ext tensorboard



Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 100376.1172 - mae: 100376.1172 - val_loss: 98514.7422 - val_mae: 98514.7422
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 99636.7422 - mae: 99636.7422 - val_loss: 97023.8984 - val_mae: 97023.8984
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 97057.8906 - mae: 97057.8906 - val_loss: 93231.1875 - val_mae: 93231.1875
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 91984.9375 - mae: 91984.9375 - val_loss: 86898.0547 - val_mae: 86898.0547
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 84549.7734 - mae: 84549.7734 - val_loss: 78660.9453 - val_mae: 78660.9453
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 75669.9844 - mae: 75669.9844 - val_loss: 69949.3203 - val_mae: 69949.3203
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 66971.7969 - mae: 66971.7969 - val_loss: 62238.1602 - val_mae: 62238.1602
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 

Reusing TensorBoard on port 6008 (pid 21800), started 0:02:37 ago. (Use '!kill 21800' to kill it.)

In [9]:
##Evaluate the loss for test data

test_loss, test_mae=model.evaluate(X_test, y_test)
print(f'Test MAE is : {test_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50302.8086 - mae: 50302.8086
Test MAE is : 50302.80859375


In [10]:
##Save regression model
model.save('regression_model.h5')